In [1]:
from src.dataset import BLEnDDataset


countries = ['China', 'Spain', 'Greece', 'Indonesia']
annotations_dir = 'data/annotations'

dataset = BLEnDDataset(countries)
dataset.load(annotations_dir)

{'China':     question_id           local_question  \
 0      Al-en-01       在中国，学龄前儿童常见的零食是什么？   
 1      Al-en-02    在中国，有什么受欢迎的食物可以搭配啤酒吃？   
 2      Al-en-04           在中国最受欢迎的水果是什么？   
 3      Al-en-06       在中国的学校食堂里常见的食物是什么？   
 4      Al-en-08     在中国的购物中心里，最常见的零食有哪些？   
 ..          ...                      ...   
 495    Th-en-53  在中国的学校里，最受欢迎的课外社交活动是什么？   
 496    Th-en-58   在中国大学之前，学的最高级的数学科目是什么？   
 497   Tmp-ar-01     中国的大学生通常去哪里复习准备期末考试？   
 498   Tmp-ar-02   在中国，人们通常乘坐什么公共交通工具去大学？   
 499   Tmp-ar-04         中国的小学生放学后通常做些什么？   
 
                                            en_question  \
 0    What is a common snack for preschool kids in C...   
 1     What is a popular food to go with beer in China?   
 2             What is the most popular fruit in China?   
 3     What is a common school cafeteria food in China?   
 4    What are the most commonly eaten snacks at sho...   
 ..                                                 ...   
 495  What is the most popular extra

In [ ]:
from src.prompts import PromptBuilder
import os
from src.LLMWrapper import LLMWrapper
import pandas as pd

builder = PromptBuilder()

model_name = 'CohereLabs/tiny-aya-global'
llm = LLMWrapper(model_name)
llm.load()

os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/outputs', exists_ok = True)

for country in countries:
    df = dataset.get_country_data(country)

    df['control_prompts'] = builder.build_control_prompts(df)
    df['persona_prompts'] = builder.build_persona_prompts(df)
    df['language_prompts'] = builder.build_language_prompts(df)


    df.to_csv(f'data/processed/{country}_prompts.csv', index = False)

    output_df = df.copy()

    output_df['control_output'] = llm.generate_batch(df['control_prompts'])
    output_df['persona_output'] = llm.generate_batch(df['persona_prompts'])
    output_df['language_output'] = llm.generate_batch(df['language_prompts'])


    output_df.to_csv(f'data/outputs/{country}_outputs.csv', index = False)
    
